# Imports & Configuration

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.config import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, classification_report

# Data Loading & Preprocessing

In [2]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "breast_cancer.parquet")
if "samples" in df.columns:
    df = df.drop(columns=["samples"])

TARGET_COL = "type"
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Dataset loaded successfully. Shape: {X.shape}")

Dataset loaded successfully. Shape: (151, 54675)


# Define the Multi-Stage Pipeline

In [3]:
# Build a joint pipeline incorporating variance filtering and selection
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=0.1)),
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Setting Up the Hyperparameter Tuning Grid

In [4]:
param_grid = {
    "feature_selection__k": [500, 1000, 2000],        
    "model__n_estimators": [100, 300, 500],          
    "model__max_depth": [10, 20, None],               
    "model__min_samples_split": [2, 5]               
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

# Executing Grid Search & Inspect Results

In [5]:
print("Starting grid search hyperparameter optimization...")
grid_search.fit(X, y_encoded)

print("\n🏆 Grid Search Optimization Complete!")
print(f"Best cross-validated F1-Score: {grid_search.best_score_:.4f}")
print("Best Hyperparameter Configuration:")
for param, val in grid_search.best_params_.items():
    print(f" -> {param}: {val}")

Starting grid search hyperparameter optimization...
Fitting 5 folds for each of 54 candidates, totalling 270 fits

🏆 Grid Search Optimization Complete!
Best cross-validated F1-Score: 0.9733
Best Hyperparameter Configuration:
 -> feature_selection__k: 500
 -> model__max_depth: 10
 -> model__min_samples_split: 2
 -> model__n_estimators: 100


# Exporting the Optimized Pipeline Artifacts

In [6]:
best_pipeline = grid_search.best_estimator_

joblib.dump(best_pipeline, MODEL_DIR / "tuned_final_model.pkl")
joblib.dump(label_encoder, ARTIFACT_DIR / "tuned_label_encoder.pkl")

cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_parquet(ARTIFACT_DIR / "grid_search_log.parquet", index=False)

print("Optimized pipeline and metadata successfully saved to project directories.")

Optimized pipeline and metadata successfully saved to project directories.
